# PPG Signal Quality Assessment and Artifact Removal Framework

## What this project does, in plain terms

A PPG sensor (the light sensor in a smartwatch or pulse oximeter) measures your heartbeat
by shining light into your skin and watching how much bounces back. The problem is: any
movement, bad lighting, or loose sensor contact adds "noise" on top of the real heartbeat
signal, and a doctor or algorithm using that signal can get fooled by the noise.

This project builds a system that:

1. **Looks at a PPG signal and decides what's wrong with it** (Is it clean? Did the
   person move? Is there bad lighting? etc.) -- 6 possible answers.
2. **Cleans the signal** using a fix tailored to whatever's wrong with it.
3. **Measures whether the cleaning worked**, by checking the cleaned result against the
   real original signal (normally impossible outside a lab, but doable here because we
   create the "noisy" version ourselves and always keep the original).
4. **Checks honestly how well all of this carries over to real recordings** that real
   experts labeled (BUT PPG), and how well it carries over to a data source the model
   never saw during training.

We use real PPG data from public medical datasets, not made-up numbers.

**How to run:** put this notebook in your project folder (the one that contains
`PPG_Dataset.csv` and the `datasets/` folder) and run all cells top to bottom. If your
data lives elsewhere, change `BASE_DIR` in the next cell. The SVM steps (5, 5b, 5c) take
the longest; everything else is quick.


In [1]:
import os, glob, pickle, time, warnings
from fractions import Fraction

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import kurtosis, skew
from scipy.signal import (butter, filtfilt, iirnotch, detrend,
                          savgol_filter, medfilt, welch, resample_poly)
from scipy.interpolate import interp1d
import pywt
from sklearn.model_selection import GroupKFold, GridSearchCV, learning_curve
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             f1_score, balanced_accuracy_score, roc_auc_score)

warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------- settings you may change -----------------------------
BASE_DIR     = '.'      # folder containing PPG_Dataset.csv and datasets/
FS           = 125      # every signal is standardized to this many samples per second
SEG_LEN      = 2000     # every segment is exactly this many samples (16 seconds)
RANDOM_STATE = 42       # fixed seed so re-running gives the same results
TEST_FRACTION = 0.2     # share of recording-groups held out for testing
SEVERITIES   = (0.7, 1.0, 1.4)   # artifact strengths used when creating practice examples

# Cuff-Less BP recordings are long (minutes) -- cap how much of it is used so it cannot
# swamp the other sources
CUFFLESS_MAX_RECORDS_PER_FILE = 300
CUFFLESS_SEGS_PER_RECORD      = 2
CUFFLESS_MAX_SEGMENTS         = 600

# BUT PPG record IDs look like 100001: the first digits identify the volunteer. The code
# checks whether grouping by this prefix gives a sensible number of volunteers, and falls
# back to one-group-per-recording if it does not.
BUT_SUBJECT_PREFIX_LEN = 3

rng = np.random.default_rng(RANDOM_STATE)   # only used as a default argument below
RESULTS = {}                                # headline numbers collected for the final summary
print("Base folder:", os.path.abspath(BASE_DIR))

Base folder: d:\VIT\ML Project


## Step 1 -- Collecting PPG signals from multiple sources

**What this does:** loads PPG signals from up to 4 public datasets and combines them into
one pool. A dataset you have not downloaded is skipped -- but **never silently**: every
loader prints exactly what it found (or why it found nothing).

**Why multiple sources:** relying on one dataset risks learning that dataset's quirks
instead of general PPG behavior.

**Sources:**

- `PPG_Dataset.csv` -- your original dataset
- Cuff-Less Blood Pressure Estimation (UCI) -- MATLAB files, read with `h5py`
- PPG-DaLiA (UCI) -- **only the "sitting still" periods**, because the rest of that
  dataset is deliberately full of real motion
- BUT PPG (PhysioNet) -- **only recordings a real expert labeled "good quality"** go
  into the training pool. The poor-quality recordings are kept aside and used _only_ for
  the real-world check in Step 9.

**Recording groups:** every segment is tagged with the recording/volunteer it came from.
Later (Step 2) the train/test split keeps each group entirely on one side, so the test
set contains people and recordings the model has never seen.


In [2]:
# ----------------------------------------------------------------------------------
# Helpers shared by all loaders
# ----------------------------------------------------------------------------------
def chunk_to_segments(sig_1d, seg_len=SEG_LEN):
    """Cut a long 1-D signal into back-to-back segments of exactly seg_len samples."""
    n_segs = len(sig_1d) // seg_len
    if n_segs == 0:
        return np.empty((0, seg_len))
    return np.asarray(sig_1d[:n_segs * seg_len], dtype=float).reshape(n_segs, seg_len)


def resample_to_125(sig_1d, original_fs):
    """Polyphase resampling to 125 Hz (fast for any length, with edge-safe padding)."""
    sig_1d = np.asarray(sig_1d, dtype=float)
    if original_fs is None or not np.isfinite(original_fs) or original_fs <= 0:
        raise ValueError(f"invalid sampling rate: {original_fs}")
    if abs(original_fs - FS) < 1e-9:
        return sig_1d
    ratio = Fraction(FS / original_fs).limit_denominator(1000)
    return resample_poly(sig_1d, ratio.numerator, ratio.denominator, padtype='line')


def finite_runs(x, min_len):
    """Contiguous stretches of x without NaN/inf (never glue separated pieces together)."""
    x = np.asarray(x, dtype=float)
    ok = np.isfinite(x)
    if ok.all():
        return [x] if len(x) >= min_len else []
    d = np.diff(np.concatenate([[0], ok.astype(int), [0]]))
    starts, ends = np.where(d == 1)[0], np.where(d == -1)[0]
    return [x[s:e] for s, e in zip(starts, ends) if e - s >= min_len]


def _empty():
    return np.empty((0, SEG_LEN)), np.array([], dtype=object)


# ----------------------------------------------------------------------------------
# Source 1: original CSV
# ----------------------------------------------------------------------------------
def load_original_csv(path):
    if not os.path.exists(path):
        print(f"[original_csv] NOT FOUND: {os.path.abspath(path)}")
        return _empty()
    df = pd.read_csv(path)
    sig_cols = [c for c in df.columns if c != 'Label']
    pool = df[sig_cols].values.astype(float)
    if pool.shape[1] != SEG_LEN:
        print(f"[original_csv] WARNING: {pool.shape[1]} columns per row, expected {SEG_LEN} -- skipped")
        return _empty()
    # The CSV has no volunteer information, so each row counts as its own group.
    groups = np.array([f"csv_{i}" for i in range(len(pool))], dtype=object)
    print(f"[original_csv] loaded {pool.shape[0]} segments")
    return pool, groups


# ----------------------------------------------------------------------------------
# Source 2: Cuff-Less Blood Pressure (MATLAB cell arrays, each cell = 3 x length matrix:
#           row 0 = PPG, row 1 = ABP, row 2 = ECG, all at 125 Hz)
# ----------------------------------------------------------------------------------
def _cuffless_cells_h5(fpath, pick_rng, max_records):
    """MATLAB v7.3 (HDF5). A MATLAB cell array is stored as a dataset of references, and
    a 3 x L matrix shows up transposed (L x 3) -- so we flatten the reference array and
    orient every record ourselves instead of assuming a layout."""
    import h5py
    out = []
    with h5py.File(fpath, 'r') as f:
        keys = [k for k in f.keys() if not k.startswith('#')]
        if not keys:
            raise ValueError("no data key in file")
        refs = np.array(f[keys[0]]).ravel()
        pick = np.sort(pick_rng.choice(len(refs), size=min(len(refs), max_records), replace=False))
        for i in pick:
            try:
                out.append((int(i), np.array(f[refs[i]], dtype=float)))
            except Exception:
                continue
    return out, len(refs)


def _cuffless_cells_v5(fpath, pick_rng, max_records):
    """Older MATLAB format (v5/v7) -- handled by scipy."""
    from scipy.io import loadmat
    m = loadmat(fpath)
    keys = [k for k in m if not k.startswith('__')]
    if not keys:
        raise ValueError("no data key in file")
    cells = np.asarray(m[keys[0]]).ravel()
    pick = np.sort(pick_rng.choice(len(cells), size=min(len(cells), max_records), replace=False))
    return [(int(i), np.asarray(cells[i], dtype=float)) for i in pick], len(cells)


def load_cuffless_bp(folder):
    if not os.path.isdir(folder):
        print(f"[cuffless_bp] folder NOT FOUND: {os.path.abspath(folder)}")
        return _empty()
    mat_files = sorted(glob.glob(os.path.join(folder, '**', '*.mat'), recursive=True))
    if not mat_files:
        print(f"[cuffless_bp] folder exists but contains NO .mat files (subfolders searched too): "
              f"{os.path.abspath(folder)}")
        print("              contents:", os.listdir(folder)[:15])
        return _empty()

    pick_rng = np.random.default_rng(RANDOM_STATE)
    segs_all, groups_all = [], []
    for fpath in mat_files:
        fname = os.path.basename(fpath)
        try:
            try:
                cells, n_total = _cuffless_cells_h5(fpath, pick_rng, CUFFLESS_MAX_RECORDS_PER_FILE)
                fmt = "v7.3/HDF5"
            except OSError:                       # not an HDF5 file -> older MATLAB format
                cells, n_total = _cuffless_cells_v5(fpath, pick_rng, CUFFLESS_MAX_RECORDS_PER_FILE)
                fmt = "v5/v7"
        except ImportError:
            print("[cuffless_bp] h5py is not installed -- run: pip install h5py   (then re-run this cell)")
            return _empty()
        except Exception as e:
            print(f"[cuffless_bp] could not read {fname}: {type(e).__name__}: {e}")
            continue

        n_ok, n_short, first_shape = 0, 0, None
        for rec_idx, rec in cells:
            if rec.ndim != 2:
                continue
            if first_shape is None:
                first_shape = rec.shape
            if rec.shape[0] > rec.shape[1]:       # (L x 3) -> (3 x L)
                rec = rec.T
            ppg = rec[0]
            chunks = []
            for run in finite_runs(ppg, SEG_LEN):
                s = chunk_to_segments(run)
                chunks.extend(list(s))
            if not chunks:
                n_short += 1
                continue
            k = min(CUFFLESS_SEGS_PER_RECORD, len(chunks))
            for j in np.unique(np.linspace(0, len(chunks) - 1, k).round().astype(int)):
                segs_all.append(chunks[j])
                groups_all.append(f"cuff_{fname}_{rec_idx}")
            n_ok += 1
        print(f"[cuffless_bp] {fname} ({fmt}): {n_total} records in file, read {len(cells)}, "
              f"usable {n_ok}, too short/empty {n_short}, first record shape {first_shape}")

    if not segs_all:
        print("[cuffless_bp] files were read but produced 0 usable segments")
        return _empty()
    segs = np.array(segs_all)
    groups = np.array(groups_all, dtype=object)
    if len(segs) > CUFFLESS_MAX_SEGMENTS:
        keep = np.sort(pick_rng.choice(len(segs), CUFFLESS_MAX_SEGMENTS, replace=False))
        segs, groups = segs[keep], groups[keep]
    print(f"[cuffless_bp] loaded {len(segs)} segments from {len(mat_files)} file(s)")
    return segs, groups


# ----------------------------------------------------------------------------------
# Source 3: PPG-DaLiA (sitting-still periods only)
# ----------------------------------------------------------------------------------
def load_ppg_dalia(folder, activity_filter=True):
    """PPG-DaLiA was deliberately recorded during real physical activity, so most of it is
    NOT clean. Only the 'sitting still' activity (code 1) is used; the activity labels run
    at 4 Hz while BVP runs at 64 Hz (ratio 16:1), so labels are repeated 16x to align them.
    Only contiguous sitting runs are used -- separated pieces are never spliced together."""
    if not os.path.isdir(folder):
        print(f"[ppg_dalia] folder NOT FOUND: {os.path.abspath(folder)}")
        return _empty()
    pkl_files = sorted(glob.glob(os.path.join(folder, '**', '*.pkl'), recursive=True))
    print(f"[ppg_dalia] found {len(pkl_files)} .pkl files")
    if not pkl_files:
        return _empty()

    SITTING_CODE, RATIO = 1, 16
    segs_all, groups_all = [], []
    total_raw, total_sitting = 0, 0
    for fpath in pkl_files:
        try:
            with open(fpath, 'rb') as f:
                data = pickle.load(f, encoding='latin1')
            ppg_raw = np.asarray(data['signal']['wrist']['BVP'], dtype=float).flatten()
            total_raw += len(ppg_raw)
            if activity_filter and 'activity' in data:
                act = np.repeat(np.asarray(data['activity']).flatten(), RATIO)[:len(ppg_raw)]
                mask = act == SITTING_CODE
                d = np.diff(mask.astype(int))
                starts, ends = np.where(d == 1)[0] + 1, np.where(d == -1)[0] + 1
                if len(mask) and mask[0]:
                    starts = np.concatenate([[0], starts])
                if len(mask) and mask[-1]:
                    ends = np.concatenate([ends, [len(mask)]])
                runs = [ppg_raw[s:e] for s, e in zip(starts, ends)]
            else:
                runs = [ppg_raw]
            subject = os.path.basename(os.path.dirname(fpath)) or os.path.splitext(os.path.basename(fpath))[0]
            for run in runs:
                total_sitting += len(run)
                for sub in finite_runs(run, 10):
                    s = chunk_to_segments(resample_to_125(sub, 64))
                    if len(s):
                        segs_all.append(s)
                        groups_all += [f"dalia_{subject}"] * len(s)
        except Exception as e:
            print(f"[ppg_dalia] skipped {os.path.basename(fpath)}: {type(e).__name__}: {e}")

    if activity_filter and total_raw > 0:
        print(f"[ppg_dalia] kept {100 * total_sitting / total_raw:.1f}% of raw recording time (sitting only)")
    if not segs_all:
        print("[ppg_dalia] 0 usable segments")
        return _empty()
    pool = np.vstack(segs_all)
    print(f"[ppg_dalia] loaded {pool.shape[0]} segments from {len(set(groups_all))} subject(s)")
    return pool, np.array(groups_all, dtype=object)


# ----------------------------------------------------------------------------------
# Source 4: BUT PPG. Returns the training pool (good-quality only) AND a table of every
# record with its expert label, used later for the real-world check.
# ----------------------------------------------------------------------------------
def load_but_ppg(folder):
    pool_empty = _empty() + (None,)
    if not os.path.isdir(folder):
        print(f"[but_ppg] folder NOT FOUND: {os.path.abspath(folder)}")
        return pool_empty
    try:
        import wfdb
    except ImportError:
        print("[but_ppg] wfdb not installed -- run: pip install wfdb")
        return pool_empty

    all_hea = glob.glob(os.path.join(folder, '**', '*.hea'), recursive=True)
    hea_files = sorted(f for f in all_hea if '_ppg' in os.path.basename(f).lower())
    print(f"[but_ppg] found {len(all_hea)} .hea files total, {len(hea_files)} are PPG records")
    if not hea_files:
        return pool_empty

    quality_map = None
    ann_candidates = glob.glob(os.path.join(folder, '**', 'quality-hr-ann.csv'), recursive=True)
    if ann_candidates:
        try:
            ann = pd.read_csv(ann_candidates[0])
            ann = ann.dropna(subset=['ID', 'Quality'])
            quality_map = dict(zip(ann['ID'].astype(str), ann['Quality'].astype(float)))
            n_good = sum(v == 1 for v in quality_map.values())
            print(f"[but_ppg] {n_good} of {len(quality_map)} annotated records are expert-labeled good quality")
        except Exception as e:
            print(f"[but_ppg] could not read quality-hr-ann.csv ({e}) -- using all recordings, no external check")
    else:
        print("[but_ppg] quality-hr-ann.csv not found -- using all recordings, no external check")

    pool_segs, pool_rids, ext_rows, diag = [], [], [], []
    n_failed, first_err = 0, None
    for hea_path in hea_files:
        rpath = hea_path[:-4]
        rid = os.path.basename(rpath).split('_')[0]
        q = quality_map.get(rid, np.nan) if quality_map is not None else np.nan
        try:
            rec = wfdb.rdrecord(rpath)
            sig = rec.p_signal if rec.p_signal is not None else rec.d_signal
            sig = np.asarray(sig, dtype=float).flatten()      # flatten handles the transposed-shape quirk
            sig = sig[np.isfinite(sig)]
            if len(sig) < 10:
                continue
            sig125 = resample_to_125(sig, rec.fs)
        except Exception as e:
            n_failed += 1
            first_err = first_err or f"{type(e).__name__}: {e}"
            continue
        segs = chunk_to_segments(sig125)
        diag.append((rec.fs, len(sig), len(sig125), q))
        ext_rows.append(dict(record_id=rid, quality=q,
                             segment=segs[0] if len(segs) else None, n_samples_125=len(sig125)))
        if (quality_map is None or q == 1) and len(segs):
            pool_segs.append(segs)
            pool_rids += [rid] * len(segs)
    if n_failed:
        print(f"[but_ppg] {n_failed} records could not be read (first error: {first_err})")
    if not ext_rows:
        return pool_empty

    d = pd.DataFrame(diag, columns=['fs', 'raw_len', 'len_125', 'quality'])
    print(f"[but_ppg] sampling rates in headers: {d['fs'].value_counts().to_dict()}")
    print(f"[but_ppg] length after resampling to 125 Hz -- min {d['len_125'].min()}, "
          f"median {int(d['len_125'].median())}, max {d['len_125'].max()} samples (segment needs {SEG_LEN})")
    if quality_map is not None:
        for qv, name in [(1, 'good'), (0, 'poor')]:
            sub = d[d['quality'] == qv]
            if len(sub):
                print(f"[but_ppg] {name}: {len(sub)} readable records, "
                      f"{(sub['len_125'] >= SEG_LEN).sum()} long enough for a full segment")

    # --- decide how to group records into volunteers ---
    rids = [r['record_id'] for r in ext_rows]
    prefixes = {r[:BUT_SUBJECT_PREFIX_LEN] for r in rids}
    use_prefix = 5 <= len(prefixes) <= 0.5 * len(rids)
    gmap = (lambda r: f"but_{r[:BUT_SUBJECT_PREFIX_LEN]}") if use_prefix else (lambda r: f"but_{r}")
    print(f"[but_ppg] grouping: {'by ID prefix -> ' + str(len(prefixes)) + ' volunteer groups' if use_prefix else 'one group per recording (ID prefix did not give a sensible number of volunteers)'}")

    ext = pd.DataFrame(ext_rows)
    ext['group'] = ext['record_id'].map(gmap)
    if not pool_segs:
        print("[but_ppg] 0 usable training segments")
        return _empty() + (ext if quality_map is not None else None,)
    pool = np.vstack(pool_segs)
    groups = np.array([gmap(r) for r in pool_rids], dtype=object)
    print(f"[but_ppg] loaded {pool.shape[0]} good-quality training segments")
    return pool, groups, (ext if quality_map is not None else None)


# ----------------------------------------------------------------------------------
# Combine everything + basic sanity screen
# ----------------------------------------------------------------------------------
def sanity_screen(segs):
    """Drop segments that are obviously broken: NaN/inf, flat-lined, or mostly repeated values."""
    finite = np.isfinite(segs).all(axis=1)
    safe = np.where(finite[:, None], segs, 0.0)
    ok = finite & (safe.std(axis=1) > 1e-9) & ((np.diff(safe, axis=1) == 0).mean(axis=1) < 0.5)
    return ok


def build_combined_pool(base_dir):
    cuff = load_cuffless_bp(os.path.join(base_dir, 'datasets', 'cuffless_bp'))
    dalia = load_ppg_dalia(os.path.join(base_dir, 'datasets', 'ppg_dalia'))
    but_segs, but_groups, but_ext = load_but_ppg(os.path.join(base_dir, 'datasets', 'but_ppg'))
    named = [
        ("original_csv", load_original_csv(os.path.join(base_dir, 'PPG_Dataset.csv'))),
        ("cuffless_bp", cuff),
        ("ppg_dalia", dalia),
        ("but_ppg", (but_segs, but_groups)),
    ]
    sig_parts, src_parts, grp_parts = [], [], []
    print("\nSanity screen (drops NaN / flat-lined / mostly-repeated-value segments):")
    for name, (segs, groups) in named:
        if segs.shape[0] == 0:
            print(f"  {name}: 0 segments -- source not used")
            continue
        ok = sanity_screen(segs)
        print(f"  {name}: kept {ok.sum()} of {len(ok)}")
        sig_parts.append(segs[ok]); src_parts.append(np.full(ok.sum(), name, dtype=object)); grp_parts.append(groups[ok])
    if not sig_parts:
        raise RuntimeError("No dataset sources found -- check BASE_DIR and the folder structure above.")
    pool = np.vstack(sig_parts)
    sources = np.concatenate(src_parts)
    groups = np.concatenate(grp_parts)
    print(f"\nTotal combined pool: {pool.shape[0]} signals")
    print("Breakdown by source:", pd.Series(sources).value_counts().to_dict())
    return pool, sources, groups, but_ext


pool_raw, pool_source_labels, pool_group_labels, but_external = build_combined_pool(BASE_DIR)
RESULTS['pool_by_source'] = pd.Series(pool_source_labels).value_counts().to_dict()

[cuffless_bp] Part_1.mat (v7.3/HDF5): 3000 records in file, read 300, usable 276, too short/empty 24, first record shape (2000, 3)
[cuffless_bp] Part_2.mat (v7.3/HDF5): 3000 records in file, read 300, usable 283, too short/empty 17, first record shape (74000, 3)
[cuffless_bp] Part_3.mat (v7.3/HDF5): 3000 records in file, read 300, usable 254, too short/empty 46, first record shape (4000, 3)
[cuffless_bp] Part_4.mat (v7.3/HDF5): 3000 records in file, read 300, usable 267, too short/empty 33, first record shape (68000, 3)
[cuffless_bp] loaded 600 segments from 4 file(s)
[ppg_dalia] found 15 .pkl files
[ppg_dalia] kept 7.1% of raw recording time (sitting only)
[ppg_dalia] loaded 564 segments from 15 subject(s)
[but_ppg] found 11616 .hea files total, 3888 are PPG records
[but_ppg] 830 of 3888 annotated records are expert-labeled good quality
[but_ppg] sampling rates in headers: {30: 3888}
[but_ppg] length after resampling to 125 Hz -- min 1250, median 3750, max 3750 samples (segment needs 

## Step 2 -- Setting aside a "test" set the model never sees during training

**What this does:** splits the signals into 80% for training and 20% for testing -- but
splits by **recording group** (a volunteer, or a recording), not by individual segment.

**Why this matters:** neighbouring segments cut from the same person's recording look very
alike. If one landed in training and its neighbour in testing, the model could score well
just by recognising that person -- not by learning anything general. Keeping every group
entirely on one side means the test set holds people/recordings the model has never seen.
(Each data source is split separately, so every source appears in both sides. The original
CSV has no volunteer information, so each of its rows counts as its own group -- a
limitation noted at the end.)

**Amplitude scaling:** different sources store PPG in very different raw units. Each source
is divided by **one scale factor (its median standard deviation), computed from the
training signals only**, so test data never influences preprocessing. One factor per
_source_ (not per signal) keeps real strong-vs-weak signal differences intact, which the
StdDev feature relies on.


In [3]:
def group_split(source_labels, group_labels, test_size=TEST_FRACTION, seed=RANDOM_STATE):
    """True for test rows. Whole groups go to one side; done separately inside each source."""
    r = np.random.default_rng(seed)
    is_test = np.zeros(len(source_labels), dtype=bool)
    for s in pd.unique(source_labels):
        idx = np.where(source_labels == s)[0]
        ug = np.unique(group_labels[idx])
        if len(ug) < 2:
            print(f"  {s}: only {len(ug)} group -- everything goes to training (cannot split)")
            continue
        r.shuffle(ug)
        n_test = max(1, int(round(test_size * len(ug))))
        test_groups = ug[:n_test]
        is_test[idx] = np.isin(group_labels[idx], test_groups)
        print(f"  {s}: {len(ug)} groups -> {len(ug) - n_test} train / {n_test} test "
              f"({(~is_test[idx]).sum()} train / {is_test[idx].sum()} test segments)")
    return is_test

print("Group-aware split:")
is_test = group_split(pool_source_labels, pool_group_labels)

train_raw, test_raw = pool_raw[~is_test], pool_raw[is_test]
train_src, test_src = pool_source_labels[~is_test], pool_source_labels[is_test]
train_grp, test_grp = pool_group_labels[~is_test], pool_group_labels[is_test]

# one scale factor per source, from TRAINING signals only
SOURCE_SCALE_FACTORS = {}
print("\nAmplitude scale per source (training signals only):")
for name in pd.unique(train_src):
    stds = train_raw[train_src == name].std(axis=1)
    SOURCE_SCALE_FACTORS[name] = max(float(np.median(stds)), 1e-6)
    print(f"  {name}: median std = {np.median(stds):.3f}  -> divided by {SOURCE_SCALE_FACTORS[name]:.3f}")

def apply_scale(signals, sources):
    factors = np.array([SOURCE_SCALE_FACTORS.get(s, 1.0) for s in sources])
    return signals / factors[:, None]

train_clean = apply_scale(train_raw, train_src)
test_clean = apply_scale(test_raw, test_src)
print(f"\nTraining signals: {len(train_clean)}   Testing signals: {len(test_clean)}")
assert len(set(train_grp) & set(test_grp)) == 0, "a group appears on both sides of the split!"
print("Check passed: no recording group appears on both sides of the split.")

Group-aware split:
  original_csv: 2576 groups -> 2061 train / 515 test (2061 train / 515 test segments)
  cuffless_bp: 525 groups -> 420 train / 105 test (480 train / 120 test segments)
  ppg_dalia: 15 groups -> 12 train / 3 test (447 train / 117 test segments)
  but_ppg: 38 groups -> 30 train / 8 test (702 train / 93 test segments)

Amplitude scale per source (training signals only):
  original_csv: median std = 0.426  -> divided by 0.426
  cuffless_bp: median std = 0.581  -> divided by 0.581
  ppg_dalia: median std = 31.415  -> divided by 31.415
  but_ppg: median std = 87.465  -> divided by 87.465

Training signals: 3690   Testing signals: 845
Check passed: no recording group appears on both sides of the split.


## Step 3 -- Creating practice examples by adding fake noise

**The problem:** to teach a model "this is what a Motion Artifact looks like," we need lots
of labeled examples. Nobody has hand-labeled thousands of PPG signals by exact artifact
type at scale.

**The solution:** take real, clean PPG signals and deliberately add a computer-generated
version of each noise type. This gives a perfect answer key -- we always know exactly what
was added, so we can also check afterwards how well the cleanup worked.

| Artifact               | What causes it in real life      | What we add to simulate it  |
| ---------------------- | -------------------------------- | --------------------------- |
| Motion Artifact        | Person moves, sensor shifts      | Sudden large, jagged bursts |
| Baseline Wander        | Breathing, slow posture changes  | A slow up-and-down drift    |
| Powerline Interference | Nearby electrical equipment      | A steady 50Hz buzz          |
| Muscle Noise           | Muscle twitching near the sensor | Random high-frequency noise |
| Ambient Light          | Room lighting changes            | A sudden jump up or down    |

Each clean signal becomes 18 examples: 3 "Clean" copies (with a tiny sensor-noise jitter so
they are not identical) and 3 strengths of each of the 5 artifacts. This keeps all 6
classes exactly equal in size.

**Important caveat, stated up front:** because the test examples are produced by the _same
noise generators_ as the training examples, a high score here measures "can the model
recognise noise like the noise it was taught," not "can it spot real-world bad
recordings." Step 5c and Step 9 test the second question directly.


In [4]:
def inject_motion_artifact(sig, severity=1.0, rng=rng):
    sig = sig.copy()
    n = len(sig)
    for _ in range(rng.integers(1, 3)):
        start = rng.integers(0, n - int(0.5 * FS))
        dur = rng.integers(int(0.15 * FS), int(0.5 * FS))
        end = min(start + dur, n)
        amp = severity * rng.uniform(1.5, 3.0) * sig.std()
        burst = amp * np.sin(np.linspace(0, rng.uniform(2, 5) * np.pi, end - start))
        burst += rng.normal(0, 0.3 * amp, end - start)
        sig[start:end] += burst
    return sig

def inject_baseline_wander(sig, severity=1.0, rng=rng):
    n = len(sig)
    t = np.arange(n) / FS
    freq = rng.uniform(0.1, 0.4)
    amp = severity * rng.uniform(1.0, 2.5) * sig.std()
    phase = rng.uniform(0, 2 * np.pi)
    return sig + amp * np.sin(2 * np.pi * freq * t + phase)

def inject_powerline_interference(sig, severity=1.0, rng=rng):
    n = len(sig)
    t = np.arange(n) / FS
    amp = severity * rng.uniform(0.15, 0.4) * sig.std()
    return sig + amp * np.sin(2 * np.pi * 50 * t + rng.uniform(0, 2 * np.pi))

def inject_muscle_noise(sig, severity=1.0, rng=rng):
    noise = rng.normal(0, severity * rng.uniform(0.3, 0.6) * sig.std(), len(sig))
    return sig + noise

def inject_ambient_light(sig, severity=1.0, rng=rng):
    sig = sig.copy()
    n = len(sig)
    for _ in range(rng.integers(1, 3)):
        idx = rng.integers(int(0.1 * n), int(0.9 * n))
        shift = severity * rng.uniform(1.0, 2.0) * sig.std() * rng.choice([-1, 1])
        sig[idx:] += shift
    return sig

def inject_clean_jitter(sig, severity=1.0, rng=rng):
    # tiny sensor-noise-floor jitter, far below any real artifact
    return sig + rng.normal(0, 0.01 * sig.std(), len(sig))

INJECTORS = {
    "Motion_Artifact": inject_motion_artifact,
    "Baseline_Wander": inject_baseline_wander,
    "Powerline_Interference": inject_powerline_interference,
    "Muscle_Noise": inject_muscle_noise,
    "Ambient_Light": inject_ambient_light,
}

def build_labeled_dataset(clean_signals, rng_seed=42, severities=SEVERITIES):
    """Returns examples, labels, and which base signal each example came from."""
    local_rng = np.random.default_rng(rng_seed)
    X_out, y_out, base_out = [], [], []
    for i, sig in enumerate(clean_signals):
        for _ in range(len(severities)):
            X_out.append(inject_clean_jitter(sig, rng=local_rng)); y_out.append("Clean"); base_out.append(i)
        for label, fn in INJECTORS.items():
            for sev in severities:
                X_out.append(fn(sig, severity=sev, rng=local_rng)); y_out.append(label); base_out.append(i)
    return np.array(X_out), np.array(y_out), np.array(base_out)

X_train_raw, y_train, base_train = build_labeled_dataset(train_clean, rng_seed=1)
X_test_raw, y_test, base_test = build_labeled_dataset(test_clean, rng_seed=2)

# bookkeeping: which recording group / source every example belongs to
groups_train, src_train_ex = train_grp[base_train], train_src[base_train]
groups_test,  src_test_ex  = test_grp[base_test],  test_src[base_test]

print(f"Training examples: {X_train_raw.shape[0]}   Testing examples: {X_test_raw.shape[0]}")
print(pd.Series(y_train).value_counts().to_dict())

Training examples: 66420   Testing examples: 15210
{'Clean': 11070, 'Motion_Artifact': 11070, 'Baseline_Wander': 11070, 'Powerline_Interference': 11070, 'Muscle_Noise': 11070, 'Ambient_Light': 11070}


## Step 4 -- Turning each signal into a handful of descriptive numbers

**The problem:** a raw PPG segment is 2,000 numbers. A model can look at all of them, but
that is slow, needs far more data, and does not explain _why_ it decided what it did.

**The solution:** calculate 7 simple statistics that summarize the signal's _shape_ -- the
"signal quality index" approach used in medical-signal research.

| Feature                    | Plain-language meaning                                   | Good at catching             |
| -------------------------- | -------------------------------------------------------- | ---------------------------- |
| Kurtosis                   | "Spikiness" -- how many extreme sudden values            | Motion artifacts             |
| Skewness                   | Lopsidedness -- is the signal biased up or down          | Ambient light jumps          |
| Amplitude Entropy          | How random/unpredictable the values are                  | Muscle noise                 |
| Zero-Crossing Rate         | How often the signal crosses its average                 | Powerline buzz, muscle noise |
| Autocorrelation            | How repetitive the signal is from one moment to the next | Motion artifacts             |
| Low-Frequency Energy Ratio | How much is slow drifting vs. fast pulsing               | Baseline wander              |
| Standard Deviation         | Overall amount of variation                              | Weak/dropout signals         |


In [5]:
def zero_crossing_rate(sig):
    signs = np.sign(sig - np.mean(sig)); signs[signs == 0] = 1
    return np.sum(np.abs(np.diff(signs)) > 0) / len(sig)

def autocorr_lag1(sig):
    sig = sig - np.mean(sig)
    denom = np.sum(sig ** 2)
    return 0.0 if denom < 1e-12 else np.sum(sig[:-1] * sig[1:]) / denom

def low_freq_energy_ratio(sig, cutoff=0.5):
    freqs, psd = welch(sig, fs=FS, nperseg=min(512, len(sig)))
    total = np.sum(psd)
    return 0.0 if total < 1e-12 else np.sum(psd[freqs <= cutoff]) / total

def amplitude_entropy(sig, n_bins=32):
    hist, _ = np.histogram(sig, bins=n_bins, density=False)
    probs = hist / hist.sum()
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

def extract_sqi_features(sig):
    sig = np.asarray(sig, dtype=float)
    return [kurtosis(sig), skew(sig), amplitude_entropy(sig),
            zero_crossing_rate(sig), autocorr_lag1(sig),
            low_freq_energy_ratio(sig), np.std(sig)]

FEATURE_NAMES = ["Kurtosis", "Skewness", "AmplitudeEntropy", "ZeroCrossingRate",
                 "AutocorrLag1", "LowFreqEnergyRatio", "StdDev"]

X_train_feat = np.array([extract_sqi_features(s) for s in X_train_raw])
X_test_feat = np.array([extract_sqi_features(s) for s in X_test_raw])
assert np.isfinite(X_train_feat).all() and np.isfinite(X_test_feat).all(), "non-finite feature values found"

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_feat)   # fitted on training data only
X_test_scaled = scaler.transform(X_test_feat)
print(f"Every signal is now just {X_train_scaled.shape[1]} numbers instead of {SEG_LEN}.")

Every signal is now just 7 numbers instead of 2000.


## Step 5 -- Choosing and training the classifier

**The job:** given those 7 numbers, decide which of the 6 classes (Clean, or one of the 5
artifact types) the signal belongs to.

### Why SVM and Decision Tree specifically?

We train **two** different models and compare them, rather than picking one blindly.

**Support Vector Machine (SVM, RBF kernel):**

- Draws _curved_ boundaries between classes. Our 7 features do not separate the 6 classes
  with straight lines -- e.g. Baseline Wander and Ambient Light both involve low-frequency
  shifts, so the boundary between them is naturally curved.
- Works well with a small number of features (7) and tens of thousands of examples.

**Decision Tree:**

- Fully interpretable -- you can read off the exact yes/no questions that led to a
  decision, which matters in a medical-adjacent application.
- Fast, and a useful sanity check: if a simple tree does almost as well as the SVM, the
  problem is not secretly relying on hidden complexity.

### Why not other models?

- **Logistic Regression / linear models:** assume straight-line class boundaries; likely to
  underperform here.
- **KNN:** slows down as data grows, and has neither the interpretability of a tree nor the
  boundary-drawing strength of an SVM.
- **Random Forest:** many trees combined -- a sensible upgrade, but a single tree already
  makes the interpretability point, and two models keep the comparison focused.
- **Deep learning (1D-CNN on the raw signal):** needs far more data and compute, and loses
  the "7 numbers and why they mattered" explanation. Overkill for a 7-feature problem.

### How the best settings are picked -- and how the winner is picked

Both models try a small grid of settings and keep what scores best in **5-fold
cross-validation where every fold keeps whole recording groups together** (`GroupKFold`).
That matters: with ordinary cross-validation, the 18 variants made from one base signal
could be split across folds, leaking information and inflating the score.
The overall winner (SVM vs. Tree) is chosen by this **cross-validation score, not by the
test score**, so the test set stays a clean, one-time exam. (To keep run time reasonable,
the SVM's settings are searched on a random sample of up to 20,000 training examples and
the winning settings are then refit on all of them.)


In [6]:
cv = GroupKFold(n_splits=5)

# The SVM settings are searched on a random sample of the training examples (an SVM's
# training time grows very quickly with data size), then the winning settings are
# refit on ALL training examples.
GRID_SUBSAMPLE = 20000
grid_idx = np.random.default_rng(RANDOM_STATE).choice(
    len(X_train_scaled), size=min(GRID_SUBSAMPLE, len(X_train_scaled)), replace=False)

svm_grid = GridSearchCV(SVC(kernel='rbf', random_state=RANDOM_STATE),
                        {'C': [1, 10, 50], 'gamma': ['scale', 0.01, 0.1]},
                        cv=cv, scoring='f1_macro', n_jobs=-1)
svm_grid.fit(X_train_scaled[grid_idx], y_train[grid_idx], groups=groups_train[grid_idx])
svm_clf = SVC(kernel='rbf', random_state=RANDOM_STATE, **svm_grid.best_params_)
svm_clf.fit(X_train_scaled, y_train)

dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE),
                       {'max_depth': [4, 6, 8, 10, 12], 'min_samples_leaf': [1, 5, 20]},
                       cv=cv, scoring='f1_macro', n_jobs=-1)
dt_grid.fit(X_train_scaled, y_train, groups=groups_train)
dt_clf = dt_grid.best_estimator_

svm_pred, dt_pred = svm_clf.predict(X_test_scaled), dt_clf.predict(X_test_scaled)

def report(name, grid, clf, pred):
    cv_mean = grid.best_score_
    cv_std = grid.cv_results_['std_test_score'][grid.best_index_]
    tr_acc = accuracy_score(y_train, clf.predict(X_train_scaled))
    te_acc = accuracy_score(y_test, pred)
    print(f"===== {name} =====")
    print(f"Best settings: {grid.best_params_}")
    print(f"Grouped cross-validation macro-F1: {cv_mean:.4f} (+/- {cv_std:.4f})")
    print(f"Training accuracy: {tr_acc:.4f}    Test accuracy: {te_acc:.4f}    "
          f"Test macro-F1: {f1_score(y_test, pred, average='macro'):.4f}")
    print(classification_report(y_test, pred))
    return dict(cv=cv_mean, cv_std=cv_std, train_acc=tr_acc, test_acc=te_acc,
                test_f1=f1_score(y_test, pred, average='macro'))

RESULTS['svm'] = report("SVM", svm_grid, svm_clf, svm_pred)
RESULTS['dt'] = report("Decision Tree", dt_grid, dt_clf, dt_pred)

if svm_grid.best_params_['C'] == max(svm_grid.param_grid['C']):
    print("NOTE: the best C sits at the edge of the grid; a larger C might do slightly better.")

if svm_grid.best_score_ >= dt_grid.best_score_:
    best_model, best_name = svm_clf, "SVM"
else:
    best_model, best_name = dt_clf, "DecisionTree"
print(f"\n{best_name} has the higher cross-validation score -- using it for the cleaning stage below.")
RESULTS['best_name'] = best_name
RESULTS['svm_params'] = svm_grid.best_params_

# accuracy on the test set, split by data source
best_pred = best_model.predict(X_test_scaled)
print(f"\nTest accuracy of {best_name} per data source:")
RESULTS['per_source_test_acc'] = {}
for s in pd.unique(src_test_ex):
    m = src_test_ex == s
    acc = accuracy_score(y_test[m], best_pred[m])
    RESULTS['per_source_test_acc'][s] = acc
    print(f"  {s}: {acc:.4f}  ({m.sum()} test examples)")

===== SVM =====
Best settings: {'C': 50, 'gamma': 'scale'}
Grouped cross-validation macro-F1: 0.9319 (+/- 0.0027)
Training accuracy: 0.9587    Test accuracy: 0.9343    Test macro-F1: 0.9347
                        precision    recall  f1-score   support

         Ambient_Light       0.84      0.91      0.88      2535
       Baseline_Wander       0.95      0.92      0.94      2535
                 Clean       0.94      0.92      0.93      2535
       Motion_Artifact       0.98      0.96      0.97      2535
          Muscle_Noise       0.95      0.95      0.95      2535
Powerline_Interference       0.95      0.94      0.94      2535

              accuracy                           0.93     15210
             macro avg       0.94      0.93      0.93     15210
          weighted avg       0.94      0.93      0.93     15210

===== Decision Tree =====
Best settings: {'max_depth': 12, 'min_samples_leaf': 5}
Grouped cross-validation macro-F1: 0.9191 (+/- 0.0109)
Training accuracy: 0.9470    T

In [7]:
class_labels = sorted(pd.Series(y_test).unique())
cm = confusion_matrix(y_test, best_pred, labels=class_labels)
fig = go.Figure(go.Heatmap(z=cm, x=class_labels, y=class_labels, colorscale='Blues',
                           text=cm, texttemplate="%{text}", showscale=False))
fig.update_layout(title=f"Confusion Matrix -- {best_name} (test set)", height=500, template="plotly_white",
                  xaxis_title="Predicted", yaxis_title="Actual")
fig.update_yaxes(autorange="reversed")
fig.show()

**Confusion Matrix** -- shows which classes the model mixes up. The diagonal (top-left to
bottom-right) is "predicted the right answer"; anything off the diagonal is a mistake and
shows exactly which two classes get confused.


In [8]:
importances = sorted(zip(FEATURE_NAMES, dt_clf.feature_importances_), key=lambda x: x[1])
names, vals = zip(*importances)
fig = go.Figure(go.Bar(x=list(vals), y=list(names), orientation='h', marker_color='steelblue'))
fig.update_layout(title="Which features mattered most to the Decision Tree",
                  xaxis_title="Importance", template="plotly_white", height=400)
fig.show()

**Feature Importance** -- shows which of the 7 numbers the Decision Tree leaned on most. A
useful honesty check: if it relied on something nonsensical, this chart would show it.


## Step 5b -- Is the model overfitting?

**Overfitting** means a model memorized its training examples instead of learning a
general rule: it scores very high on data it has seen and noticeably lower on new data.
Three checks, all computed below:

1. **Training vs. cross-validation vs. test accuracy.** A large gap between training and
   the others is the classic warning sign.
2. **A learning curve.** Accuracy on training vs. unseen data as the amount of training
   data grows. Two lines that converge = healthy. A wide gap that does not close = overfit.
3. **Step 5c (next)** tests something harder: a data source the model never saw at all.


In [9]:
rows = []
for name, key in [("SVM", 'svm'), ("Decision Tree", 'dt')]:
    r = RESULTS[key]
    rows.append({"model": name, "train_accuracy": r['train_acc'], "cv_macro_F1": r['cv'],
                 "test_accuracy": r['test_acc'], "train_minus_test": r['train_acc'] - r['test_acc']})
gap_table = pd.DataFrame(rows).set_index("model").round(4)
print(gap_table)

gap = RESULTS[{"SVM": 'svm', "DecisionTree": 'dt'}[best_name]]['train_acc'] - \
      RESULTS[{"SVM": 'svm', "DecisionTree": 'dt'}[best_name]]['test_acc']
if gap > 0.05:
    verdict = "LARGE gap -> likely overfitting"
elif gap > 0.02:
    verdict = "moderate gap -> mild overfitting"
else:
    verdict = "small gap -> no sign of overfitting on held-out recordings"
print(f"\n{best_name}: training accuracy minus test accuracy = {gap:+.4f}  ->  {verdict}")
RESULTS['overfit_gap'], RESULTS['overfit_verdict'] = gap, verdict

# ---- learning curve (SVM, on a sample of the training data to keep it quick) ----
sub = np.random.default_rng(0).choice(len(X_train_scaled), size=min(12000, len(X_train_scaled)), replace=False)
n_groups_sub = len(np.unique(groups_train[sub]))
n_splits_lc = min(3, n_groups_sub)
sizes, tr_sc, va_sc = learning_curve(
    SVC(kernel='rbf', random_state=RANDOM_STATE, **svm_grid.best_params_),
    X_train_scaled[sub], y_train[sub], groups=groups_train[sub],
    cv=GroupKFold(n_splits=n_splits_lc), train_sizes=np.linspace(0.2, 1.0, 5),
    scoring='accuracy', n_jobs=-1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=sizes, y=tr_sc.mean(axis=1), mode='lines+markers', name='Training accuracy', line=dict(color='salmon')))
fig.add_trace(go.Scatter(x=sizes, y=va_sc.mean(axis=1), mode='lines+markers', name='Accuracy on unseen recording groups', line=dict(color='seagreen')))
fig.update_layout(title="Learning curve (SVM)", xaxis_title="Number of training examples",
                  yaxis_title="Accuracy", template="plotly_white", height=420)
fig.show()
lc_gap = tr_sc.mean(axis=1)[-1] - va_sc.mean(axis=1)[-1]
print(f"At the largest size, training minus unseen-group accuracy = {lc_gap:+.4f}")
RESULTS['learning_curve_gap'] = float(lc_gap)

               train_accuracy  cv_macro_F1  test_accuracy  train_minus_test
model                                                                      
SVM                    0.9587       0.9319         0.9343            0.0244
Decision Tree          0.9470       0.9191         0.9141            0.0328

SVM: training accuracy minus test accuracy = +0.0244  ->  moderate gap -> mild overfitting


At the largest size, training minus unseen-group accuracy = +0.0327


## Step 5c -- The harder test: a data source the model has never seen

Everything so far tests on _new recordings from the same sources_. This step trains on all
sources **except one** and tests on the one left out ("leave-one-source-out"). It is the
closest thing in this project to the question "would this work on a different sensor?"

How to read it: compare each row's accuracy with that source's normal test accuracy from
Step 5. A big drop means the model leans on quirks of the sources it trained on.
`clean_recall` is the share of that source's genuinely clean signals still called "Clean".

(One caveat: the held-out source is scaled by its own median standard deviation, which
assumes you have a batch of its data to measure. On a brand-new device you would need a
calibration step to get this scale factor.)


In [10]:
LOSO_MAX_TRAIN = 20000   # subsample for speed
X_all = np.vstack([X_train_feat, X_test_feat])
y_all = np.concatenate([y_train, y_test])
src_all = np.concatenate([src_train_ex, src_test_ex])

loso_rows = []
loso_rng = np.random.default_rng(1)
if len(pd.unique(src_all)) < 2:
    print("Only one data source available -- leave-one-source-out needs at least two.")
else:
    for held in pd.unique(src_all):
        tr_idx = np.where(src_all != held)[0]
        te_idx = np.where(src_all == held)[0]
        if len(tr_idx) > LOSO_MAX_TRAIN:
            tr_idx = loso_rng.choice(tr_idx, LOSO_MAX_TRAIN, replace=False)
        sc = StandardScaler().fit(X_all[tr_idx])
        clf = SVC(kernel='rbf', random_state=RANDOM_STATE, **svm_grid.best_params_)
        clf.fit(sc.transform(X_all[tr_idx]), y_all[tr_idx])
        pred = clf.predict(sc.transform(X_all[te_idx]))
        is_clean = y_all[te_idx] == 'Clean'
        loso_rows.append({
            "held_out_source": held, "n_examples": len(te_idx),
            "accuracy_on_unseen_source": accuracy_score(y_all[te_idx], pred),
            "macro_F1": f1_score(y_all[te_idx], pred, average='macro'),
            "clean_recall": float((pred[is_clean] == 'Clean').mean()),
            "normal_test_accuracy_same_source": RESULTS['per_source_test_acc'].get(held, np.nan)})
    loso_table = pd.DataFrame(loso_rows).set_index("held_out_source").round(4)
    print(loso_table)
    RESULTS['loso'] = loso_table
    RESULTS['loso_mean_acc'] = float(loso_table['accuracy_on_unseen_source'].mean())

                 n_examples  accuracy_on_unseen_source  macro_F1  \
held_out_source                                                    
original_csv          46368                     0.5027    0.4272   
cuffless_bp           10800                     0.4014    0.3884   
ppg_dalia             10152                     0.3877    0.3909   
but_ppg               14310                     0.3552    0.3006   

                 clean_recall  normal_test_accuracy_same_source  
held_out_source                                                  
original_csv           0.0000                            0.9830  
cuffless_bp            0.0050                            0.8681  
ppg_dalia              0.3097                            0.7559  
but_ppg                0.0000                            0.9743  


## Step 6 -- Cleaning the signal based on what was detected

**What this does:** once we know _what kind_ of problem a signal has, we apply a fix
suited to that problem -- not one generic filter for everything.

| Detected as...         | What we do about it                                           | Why this specific fix                                                                                |
| ---------------------- | ------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------- |
| Motion Artifact        | Remove extreme spikes, then wavelet smoothing                 | Spikes need direct removal; wavelet smoothing cleans the rest without flattening the heartbeat shape |
| Baseline Wander        | A strong "ignore slow changes" filter, then light smoothing   | Removes the slow drift, keeps the faster heartbeat rhythm                                            |
| Powerline Interference | A "notch" filter at exactly 50Hz                              | Removes just the interference frequency                                                              |
| Muscle Noise           | A "smooth out anything too fast" filter, then light smoothing | Muscle noise is high-frequency; a heartbeat is not                                                   |
| Ambient Light          | Recenter the signal, smooth, then remove leftover spikes      | Fixes the sudden level jump                                                                          |
| Clean                  | Nothing extra -- just a light baseline correction             | Over-processing clean signals only damages them                                                      |

**Fix in this version -- wavelet noise estimate:** the wavelet smoother decides how
aggressively to shrink detail by first estimating the noise level. The standard method
(Donoho & Johnstone) measures it from the _finest_ detail band. The earlier code measured
it from the _coarsest_ band, which contains real heartbeat energy, so the noise level was
over-estimated and the smoothing was stronger than intended. It now uses the finest band.


In [11]:
def mad(x):
    """Median absolute deviation, scaled to match a standard deviation for Gaussian noise."""
    x = np.asarray(x, dtype=float)
    return np.median(np.abs(x - np.median(x))) / 0.6745

def butter_filter(sig, cutoff, order, btype):
    nyq = 0.5 * FS
    cutoff = cutoff / nyq if isinstance(cutoff, float) else [c / nyq for c in cutoff]
    b, a = butter(order, cutoff, btype=btype)
    return filtfilt(b, a, sig)

def highpass_filter(sig, cutoff=0.3, order=4):
    return butter_filter(sig, cutoff, order, 'high')

def lowpass_filter(sig, cutoff=8.0, order=4):
    return butter_filter(sig, cutoff, order, 'low')

def notch_filter(sig, freq=50.0, Q=30.0):
    b, a = iirnotch(freq / (FS / 2), Q)
    return filtfilt(b, a, sig)

def wavelet_denoise(sig, wavelet='db6', level=4):
    max_level = pywt.dwt_max_level(len(sig), pywt.Wavelet(wavelet).dec_len)
    level = min(level, max_level) if max_level > 0 else 1
    coeffs = pywt.wavedec(sig, wavelet, level=level)
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745          # finest detail band (the fix)
    uthresh = sigma * np.sqrt(2 * np.log(len(sig)))
    coeffs[1:] = [pywt.threshold(c, value=uthresh, mode='soft') for c in coeffs[1:]]
    return pywt.waverec(coeffs, wavelet)[:len(sig)]

def median_filter(sig):
    return medfilt(sig, kernel_size=3)

def savgol_smooth(sig, window=81, polyorder=3):
    window = min(window, len(sig) if len(sig) % 2 == 1 else len(sig) - 1)
    window = max(window, polyorder + 1 + (polyorder % 2 == 0))
    if window % 2 == 0:
        window += 1
    return savgol_filter(sig, window, polyorder)

def spike_suppressor(sig, window_size=11, n_sigmas=3):
    sig = sig.copy()
    s = pd.Series(sig)
    rm = s.rolling(window_size, center=True).median()
    rmad = s.rolling(window_size, center=True).apply(mad, raw=True)
    thresh = n_sigmas * rmad.fillna(rmad.median())
    diff = np.abs(s - rm)
    mask = (diff > thresh).values & rm.notna().values
    sig[mask] = rm.values[mask]
    return sig

def despike_global(sig, n_sigmas=2.5, pad=4):
    sig = sig.copy()
    med, m = np.median(sig), mad(sig)
    if m < 1e-9:
        m = np.std(sig) + 1e-9
    mask = (np.abs(sig - med) / m) > n_sigmas
    dil = mask.copy()
    for i in np.where(mask)[0]:
        lo, hi = max(0, i - pad), min(len(sig), i + pad + 1)
        dil[lo:hi] = True
    if dil.any() and (~dil).sum() >= 2:                      # need at least 2 good points to interpolate
        idx = np.arange(len(sig))
        f = interp1d(idx[~dil], sig[~dil], bounds_error=False, fill_value='extrapolate')
        sig[dil] = f(idx[dil])
    return sig

def clean_ppg(sig, predicted_label):
    sig = detrend(np.asarray(sig, dtype=float))
    cleaned = highpass_filter(sig, cutoff=0.3)
    if predicted_label == 'Motion_Artifact':
        return wavelet_denoise(despike_global(cleaned))
    elif predicted_label == 'Baseline_Wander':
        cleaned = highpass_filter(cleaned, cutoff=0.6, order=8)
        cleaned = median_filter(cleaned)
        return savgol_smooth(cleaned)
    elif predicted_label == 'Ambient_Light':
        cleaned = cleaned - np.mean(cleaned)
        cleaned = median_filter(cleaned)
        cleaned = savgol_smooth(cleaned)
        return spike_suppressor(cleaned)
    elif predicted_label == 'Powerline_Interference':
        return notch_filter(cleaned)
    elif predicted_label == 'Muscle_Noise':
        cleaned = lowpass_filter(cleaned, cutoff=8.0)
        cleaned = median_filter(cleaned)
        return savgol_smooth(cleaned)
    else:
        return cleaned

# ---- the two reference signals used to judge cleaning (see Step 7) ----
def minimal_reference(true_clean_sig):
    """The true signal with ONLY the baseline correction every class gets. Class-independent."""
    return highpass_filter(detrend(np.asarray(true_clean_sig, dtype=float)), cutoff=0.3)

def matched_reference(true_clean_sig, predicted_label):
    """What the pipeline would output if handed the artifact-free signal. Built by calling
    clean_ppg itself, so it can never drift out of sync with the real cleaning chain."""
    return clean_ppg(true_clean_sig, predicted_label)

## Step 7 -- Measuring whether the cleaning actually worked

**How we check:** because we added the noise ourselves, the true original is always on
hand. For each test signal we compare the noisy version and the cleaned version against it.

**The 3 numbers:**

- **Correlation** -- similarity of shape (1.0 = identical). We want it to go UP.
- **RMSE** -- average size of the gap between two signals. We want it to go DOWN.
- **SNR (dB)** -- real signal vs. leftover noise. We want it to go UP.

**Two references, reported side by side** (this is deliberate):

1. **vs. the true signal (primary).** Before and after cleaning are judged against the
   _same_ target: the true signal with only the baseline correction that every class gets.
   This counts any damage the cleaning does to the real heartbeat shape.
2. **vs. a matched reference (secondary).** The true signal pushed through the same
   cleaning chain. This isolates "how much _artifact_ is left," ignoring the shape change
   the filters themselves cause. Built by calling `clean_ppg` itself, which fixes a bug
   class from earlier versions where a hand-written reference drifted out of sync with the
   real cleaning code (it was missing the smoothing steps for several classes).

Cleaning is applied using the **predicted** label, so any misclassification is counted
against it -- exactly what would happen in real use.

**What to expect:** the primary numbers will look more modest than the matched ones. That is
not a bug -- the smoothing filters (for example the 81-sample Savitzky-Golay step) also
smooth away some genuine fine detail of the heartbeat, and the primary reference counts
that loss while the matched reference does not. The "Clean" class shows a small degradation
because about 8% of Clean signals are misclassified as artifacts and then smoothed; this is a cost of
classification errors, not of the filters.


In [12]:
def corr(a, b):
    return 0.0 if a.std() < 1e-9 or b.std() < 1e-9 else float(np.corrcoef(a, b)[0, 1])

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def snr_db(clean_ref, signal):
    noise = signal - clean_ref
    p_sig, p_noise = np.mean(clean_ref ** 2), np.mean(noise ** 2)
    return 100.0 if p_noise < 1e-12 else float(10 * np.log10(p_sig / p_noise))

pred_labels = best_model.predict(X_test_scaled)
eval_rng = np.random.default_rng(0)
sample_idx = eval_rng.choice(len(X_test_raw), size=min(2500, len(X_test_raw)), replace=False)

rows = []
for i in sample_idx:
    raw, true_sig, pred = X_test_raw[i], test_clean[base_test[i]], pred_labels[i]
    cleaned = clean_ppg(raw, pred)
    ref_truth, ref_match = minimal_reference(true_sig), matched_reference(true_sig, pred)
    n = min(len(raw), len(cleaned), len(ref_truth), len(ref_match))
    raw_c, true_c = (raw - raw.mean())[:n], (true_sig - true_sig.mean())[:n]
    cleaned, ref_truth, ref_match = cleaned[:n], ref_truth[:n], ref_match[:n]
    rows.append({
        "true_label": y_test[i], "pred_label": pred,
        "corr_before": corr(raw_c, true_c),     "corr_after": corr(cleaned, ref_truth),     "corr_after_matched": corr(cleaned, ref_match),
        "rmse_before": rmse(raw_c, true_c),     "rmse_after": rmse(cleaned, ref_truth),     "rmse_after_matched": rmse(cleaned, ref_match),
        "snr_before":  snr_db(true_c, raw_c),   "snr_after":  snr_db(ref_truth, cleaned),   "snr_after_matched":  snr_db(ref_match, cleaned),
    })

res = pd.DataFrame(rows)
summary = res.groupby("true_label").agg(
    n=("true_label", "count"),
    corr_before=("corr_before", "mean"), corr_after=("corr_after", "mean"), corr_after_matched=("corr_after_matched", "mean"),
    rmse_before=("rmse_before", "mean"), rmse_after=("rmse_after", "mean"), rmse_after_matched=("rmse_after_matched", "mean"),
    snr_before=("snr_before", "mean"), snr_after=("snr_after", "mean"), snr_after_matched=("snr_after_matched", "mean"),
).round(4)
pd.set_option("display.width", 200)
print(summary.T)
res.to_csv("restoration_results.csv", index=False)
summary.to_csv("restoration_summary.csv")

def judge(before, after, higher_is_better, tol):
    d = (after - before) if higher_is_better else (before - after)
    return "better" if d > tol else ("worse" if d < -tol else "same")

print("\nPlain-language reading (primary comparison, same target before and after;")
print("'same' = change too small to matter: correlation < 0.005, RMSE < 3%, SNR < 0.5 dB):")
RESULTS['restoration'] = {}
for cls in summary.index:
    s = summary.loc[cls]
    j = {"correlation": judge(s.corr_before, s.corr_after, True, 0.005),
         "RMSE": judge(s.rmse_before, s.rmse_after, False, 0.03 * max(s.rmse_before, 1e-9)),
         "SNR": judge(s.snr_before, s.snr_after, True, 0.5)}
    RESULTS['restoration'][cls] = j
    print(f"  {cls:24s} correlation {j['correlation']:6s} ({s.corr_before:.3f}->{s.corr_after:.3f}) | "
          f"RMSE {j['RMSE']:6s} ({s.rmse_before:.3f}->{s.rmse_after:.3f}) | "
          f"SNR {j['SNR']:6s} ({s.snr_before:.1f}->{s.snr_after:.1f} dB)")

true_label          Ambient_Light  Baseline_Wander     Clean  Motion_Artifact  Muscle_Noise  Powerline_Interference
n                        415.0000         401.0000  395.0000         424.0000      412.0000                453.0000
corr_before                0.7635           0.6348    1.0000           0.9478        0.9039                  0.9785
corr_after                 0.8334           0.7214    0.9941           0.9546        0.8431                  0.9897
corr_after_matched         0.9601           0.9955    0.9999           0.9720        0.9723                  0.9994
rmse_before                0.8863           1.2773    0.0103           0.3126        0.4554                  0.1995
rmse_after                 0.3720           0.5701    0.0417           0.2598        0.3396                  0.0516
rmse_after_matched         0.1698           0.0241    0.0098           0.1976        0.1180                  0.0244
snr_before                 2.2742          -1.3310   40.0066          10

In [13]:
cats = list(summary.index)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Correlation to true signal", "SNR (dB, capped at 20 for readability)"))
fig.add_trace(go.Bar(name='Before cleaning', x=cats, y=summary['corr_before'], marker_color='salmon'), row=1, col=1)
fig.add_trace(go.Bar(name='After cleaning', x=cats, y=summary['corr_after'], marker_color='seagreen'), row=1, col=1)
fig.add_trace(go.Bar(x=cats, y=summary['snr_before'].clip(upper=20), marker_color='salmon', showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=cats, y=summary['snr_after'].clip(upper=20), marker_color='seagreen', showlegend=False), row=1, col=2)
fig.update_layout(barmode='group', height=450, template="plotly_white",
                  title_text="Before vs. after cleaning, by true artifact type (same reference both sides)")
fig.update_xaxes(tickangle=45)
fig.show()

## Step 8 -- See it for yourself: Noisy vs. Cleaned

This takes one real test signal, adds a Baseline Wander artifact, runs it through the full
pipeline (classify, then clean), and shows before and after. The demo picks a signal the
model classifies correctly, so you see the cleaning working as designed; if none of the
candidates is classified correctly, it says so.

**Both panels use the SAME vertical scale on purpose.** Letting each panel auto-scale would
hide a real reduction in noise, because a smaller amount of noise can look just as busy if
it fills its own panel. The grey line is the true signal, for comparison.

**Why the cleaned signal will not look like a textbook-perfect smooth wave:** real PPG
signals, even the untouched originals, have small bumps and irregular texture. A glassy
output is only possible by _over_-smoothing, which erases real detail and scores worse in
Step 7.


In [14]:
def find_correctly_classified_demo(candidate_indices, artifact_type="Baseline_Wander", severity=1.0):
    last = None
    for attempt, idx in enumerate(candidate_indices):
        ref = test_clean[idx]
        noisy = INJECTORS[artifact_type](ref, severity=severity, rng=np.random.default_rng(attempt))
        feat = scaler.transform(np.array(extract_sqi_features(noisy)).reshape(1, -1))
        pred = best_model.predict(feat)[0]
        last = (idx, ref, noisy, pred)
        if pred == artifact_type:
            return last, True
    return last, False

demo_true_label = "Baseline_Wander"
(demo_idx, demo_ref, demo_noisy, demo_pred), demo_ok = find_correctly_classified_demo(
    list(range(min(40, len(test_clean)))), artifact_type=demo_true_label)
if not demo_ok:
    print(f"NOTE: none of the first candidates was classified as {demo_true_label}; showing the last one tried "
          f"(predicted: {demo_pred}).")

demo_cleaned = clean_ppg(demo_noisy, demo_pred)
m = min(len(demo_cleaned), len(demo_ref))
demo_ref_truth = minimal_reference(demo_ref)[:m]
demo_noisy_c, demo_ref_c = (demo_noisy - demo_noisy.mean())[:m], (demo_ref - demo_ref.mean())[:m]

y_min = min(demo_noisy_c.min(), demo_cleaned[:m].min(), demo_ref_c.min())
y_max = max(demo_noisy_c.max(), demo_cleaned[:m].max(), demo_ref_c.max())
y_pad = 0.05 * (y_max - y_min)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Noisy PPG", "Cleaned PPG"), horizontal_spacing=0.08)
fig.add_trace(go.Scatter(y=demo_ref_c, mode='lines', line=dict(color='lightgrey', width=2.2), name='True signal'), row=1, col=1)
fig.add_trace(go.Scatter(y=demo_noisy_c, mode='lines', line=dict(color='black', width=1.4), name='Noisy'), row=1, col=1)
fig.add_trace(go.Scatter(y=demo_ref_truth, mode='lines', line=dict(color='lightgrey', width=2.2), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(y=demo_cleaned[:m], mode='lines', line=dict(color='black', width=1.4), name='Cleaned'), row=1, col=2)
fig.update_layout(height=450, template="plotly_white", showlegend=True,
                  title_text=f"True artifact: {demo_true_label}  |  Predicted: {demo_pred}")
fig.update_xaxes(title_text="Sample", showgrid=False)
fig.update_yaxes(showgrid=False, range=[y_min - y_pad, y_max + y_pad])
for ann in fig['layout']['annotations']:
    ann['font'] = dict(size=20)
fig.show()

print(f"Correlation to the true signal -- before: {corr(demo_noisy_c, demo_ref_c):.3f}, after: {corr(demo_cleaned[:m], demo_ref_truth):.3f}")
print(f"RMSE to the true signal        -- before: {rmse(demo_noisy_c, demo_ref_c):.3f}, after: {rmse(demo_cleaned[:m], demo_ref_truth):.3f}")
print("(Higher correlation and lower RMSE both mean the cleaned signal is genuinely closer to the truth.)")

Correlation to the true signal -- before: 0.641, after: 0.821
RMSE to the true signal        -- before: 1.018, after: 0.515
(Higher correlation and lower RMSE both mean the cleaned signal is genuinely closer to the truth.)


## Step 8b (optional) -- A presentation-friendly, amplitude-normalized view

Everything above is the real, unmodified result. This section adds a **purely cosmetic,
clearly-labeled display option**: it evens out slow amplitude changes across the window so
a signal looks more uniform at a glance, without changing its shape, timing, or any metric
from Step 7. It does NOT make the cleaning better -- it is like adjusting contrast on a
photo. Use it only for slides, and say so if asked.


In [15]:
def normalize_for_display(sig, window_sec=3.0):
    """Evens out SLOW amplitude changes only (window of several seconds). Display only."""
    window = int(window_sec * FS)
    if window % 2 == 0:
        window += 1
    s = pd.Series(sig)
    local_std = s.rolling(window, center=True, min_periods=window // 4).std().bfill().ffill()
    local_std = np.maximum(local_std.values, 0.2 * np.median(local_std.values))
    return sig * (local_std.mean() / local_std)

demo_cleaned_display = normalize_for_display(demo_cleaned[:m])
fig = go.Figure(go.Scatter(y=demo_cleaned_display, mode='lines', line=dict(color='black', width=1.4)))
fig.update_layout(height=400, template="plotly_white", showlegend=False,
                  title_text="Cleaned PPG -- display-normalized (for presentation only)")
fig.update_xaxes(title_text="Sample", showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()
print("Reminder: this view is for presentation readability only. The evidence is Step 7's numbers.")

Reminder: this view is for presentation readability only. The evidence is Step 7's numbers.


## Step 9 -- Checking against real, expert-labeled recordings

Everything above uses noise WE added on purpose. The fair question is: does this work on
completely real recordings judged by real experts? BUT PPG provides exactly that: each
recording is labeled "good" or "poor" quality by a human expert.

**What is tested, and how it is kept honest:**

- Only recordings the model **never trained on** are used for the main test: all the
  expert-labeled _poor_ recordings (never in the training pool) plus the _good_
  recordings from the held-out test groups. (An earlier version also scored recordings
  that were in the training pool.)
- Recordings are processed **exactly like training data**: same resampling, same
  per-source scale factor, and the same single 16-second segment.
- Because poor recordings outnumber good ones, plain "agreement" is misleading: a model
  that always says "not clean" would score highly. So results are reported as
  **sensitivity** (good recordings called Clean), **specificity** (poor recordings called
  not-Clean), **balanced accuracy** (their average; 50% = no better than guessing), and
  the trivial baseline.

**Part A** tests the artifact-trained model from above. **Part B** asks a different
question: _are the 7 features themselves informative about real quality?_ It trains a
simple good-vs-poor classifier on real expert labels (on recordings from some volunteers)
and tests it on other volunteers. If Part B works but Part A does not, the features are
fine and the gap is between _synthetic_ and _real_ noise.


In [16]:
RESULTS['external_A'] = RESULTS['external_B'] = None
BUT_TEST_GROUPS = set(test_grp[test_src == 'but_ppg'])

if but_external is None or len(but_external) == 0:
    print("BUT PPG expert labels not available -- real-world check skipped.")
else:
    ext = but_external.dropna(subset=['quality']).copy()
    ext['has_seg'] = ext['segment'].apply(lambda s: s is not None)
    print("Records with an expert label:", len(ext))
    for qv, nm in [(1, 'good'), (0, 'poor')]:
        sub = ext[ext['quality'] == qv]
        print(f"  {nm}: {len(sub)} total, {sub['has_seg'].sum()} long enough for a full {SEG_LEN}-sample segment, "
              f"{(~sub['has_seg']).sum()} too short (excluded)")
    valid = ext[ext['has_seg']].copy().reset_index(drop=True)

    if valid['quality'].nunique() < 2 or len(valid) < 20:
        print("Not enough labeled recordings of both kinds to evaluate.")
    else:
        but_scale = SOURCE_SCALE_FACTORS.get('but_ppg', 1.0)
        X_ext = np.array([extract_sqi_features(s / but_scale) for s in valid['segment']])
        valid['pred'] = best_model.predict(scaler.transform(X_ext))
        valid['true_good'] = valid['quality'] == 1
        valid['pred_clean'] = valid['pred'] == 'Clean'

        def binary_report(df, title):
            g, p = df['true_good'].values, df['pred_clean'].values
            sens = p[g].mean() if g.any() else np.nan
            spec = (~p[~g]).mean() if (~g).any() else np.nan
            bal = np.nanmean([sens, spec])
            acc = (g == p).mean()
            maj = max(g.mean(), 1 - g.mean())
            print(f"{title}  (n={len(df)}: {g.sum()} good, {(~g).sum()} poor)")
            print(f"  sensitivity (good called Clean):       {sens:.1%}")
            print(f"  specificity (poor called not-Clean):   {spec:.1%}")
            print(f"  balanced accuracy:                     {bal:.1%}   (50% = guessing)")
            print(f"  plain agreement:                       {acc:.1%}   (always-guess-majority baseline: {maj:.1%})")
            return dict(sensitivity=sens, specificity=spec, balanced_accuracy=bal, agreement=acc, majority_baseline=maj, n=len(df))

        # ---------- Part A ----------
        unseen = (~valid['true_good']) | (valid['group'].isin(BUT_TEST_GROUPS))
        print("\n===== PART A: artifact-trained model on real expert labels (never-trained-on recordings only) =====")
        RESULTS['external_A'] = binary_report(valid[unseen], "Part A")
        print("\nWhat the model called each real recording (rows = expert label):")
        print(pd.crosstab(valid.loc[unseen, 'true_good'].map({True: 'expert: good', False: 'expert: poor'}),
                          valid.loc[unseen, 'pred']))

        # ---------- Part B ----------
        print("\n===== PART B: are the 7 features informative about REAL quality? =====")
        pool_but_groups = set(pool_group_labels[pool_source_labels == 'but_ppg'])
        poor_only = sorted(set(valid['group']) - pool_but_groups)
        poor_only = list(np.random.default_rng(RANDOM_STATE).permutation(poor_only)) if poor_only else []
        extra_test = set(poor_only[:max(1, int(round(TEST_FRACTION * len(poor_only))))]) if poor_only else set()
        is_te = valid['group'].isin(BUT_TEST_GROUPS | extra_test).values
        y_b = valid['true_good'].values
        if is_te.sum() < 10 or len(np.unique(y_b[is_te])) < 2 or len(np.unique(y_b[~is_te])) < 2:
            print("Not enough recordings of both kinds on both sides of a volunteer-level split -- Part B skipped.")
        else:
            sc_b = StandardScaler().fit(X_ext[~is_te])
            clf_b = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', random_state=RANDOM_STATE)
            clf_b.fit(sc_b.transform(X_ext[~is_te]), y_b[~is_te])
            pred_b = clf_b.predict(sc_b.transform(X_ext[is_te]))
            auc_b = roc_auc_score(y_b[is_te], clf_b.decision_function(sc_b.transform(X_ext[is_te])))
            bal_b = balanced_accuracy_score(y_b[is_te], pred_b)
            same_A = valid[is_te]
            print(f"Trained on {(~is_te).sum()} real recordings, tested on {is_te.sum()} recordings from other volunteers/groups.")
            print(f"  Part B (trained on real labels): balanced accuracy {bal_b:.1%},  ROC-AUC {auc_b:.3f}   (AUC 0.5 = guessing, 1.0 = perfect)")
            a_same = binary_report(same_A, "  Part A on the very same test recordings, for a like-for-like comparison")
            RESULTS['external_B'] = dict(balanced_accuracy=bal_b, auc=auc_b, partA_same_records=a_same['balanced_accuracy'])
            if max(bal_b, a_same['balanced_accuracy']) < 0.70:
                print("\nReading: neither approach is strong. On these real recordings the 7 features carry only weak")
                print("information about quality, so the gap is not explained by synthetic-vs-real noise alone.")
            elif bal_b - a_same['balanced_accuracy'] > 0.10:
                print("\nReading: the 7 features DO carry real-quality information (Part B beats Part A clearly), but the")
                print("model trained only on synthetic noise does not transfer. The gap is synthetic-vs-real noise.")
            else:
                print("\nReading: Part B is not clearly better than Part A on these records.")

Records with an expert label: 3888
  good: 830 total, 795 long enough for a full 2000-sample segment, 35 too short (excluded)
  poor: 3058 total, 3045 long enough for a full 2000-sample segment, 13 too short (excluded)

===== PART A: artifact-trained model on real expert labels (never-trained-on recordings only) =====
Part A  (n=3138: 93 good, 3045 poor)
  sensitivity (good called Clean):       97.8%
  specificity (poor called not-Clean):   17.1%
  balanced accuracy:                     57.5%   (50% = guessing)
  plain agreement:                       19.5%   (always-guess-majority baseline: 97.0%)

What the model called each real recording (rows = expert label):
pred          Ambient_Light  Baseline_Wander  Clean  Motion_Artifact  Muscle_Noise  Powerline_Interference
true_good                                                                                                 
expert: good              1                0     91                1             0                       0
expert:

## Step 10 -- How confident are we in these numbers?

A single accuracy number does not say how _reliable_ it is. Two checks:

1. **Bootstrap confidence interval.** Resample the test set many times and recompute
   accuracy to get a realistic range. Resampling is done over **base signals**, not
   individual examples, because the 18 examples made from one base signal are not
   independent -- treating them as independent would make the interval too narrow.
2. **SVM vs. Decision Tree.** The same resampling is applied to the _difference_ in
   accuracy. If the interval for the difference excludes 0, the gap is unlikely to be luck.
   (This replaces McNemar's test, which also assumes independent test cases.)

Limitation: signals from the same volunteer are also correlated, and the number of
independent volunteers in the test set is small, so even these intervals are somewhat
optimistic.


In [17]:
df_b = pd.DataFrame({'base': base_test, 'svm': svm_pred == y_test, 'dt': dt_pred == y_test})
g = df_b.groupby('base').agg(n=('svm', 'size'), svm=('svm', 'sum'), dt=('dt', 'sum'))
n_arr, s_arr, d_arr = g['n'].values, g['svm'].values, g['dt'].values

boot = np.random.default_rng(0)
svm_b, dt_b = [], []
for _ in range(2000):
    idx = boot.integers(0, len(g), len(g))
    svm_b.append(s_arr[idx].sum() / n_arr[idx].sum())
    dt_b.append(d_arr[idx].sum() / n_arr[idx].sum())
svm_b, dt_b = np.array(svm_b), np.array(dt_b)
diff_b = svm_b - dt_b

def ci(a): return np.percentile(a, [2.5, 97.5])
print(f"SVM test accuracy:           {s_arr.sum() / n_arr.sum():.4f}   95% CI {ci(svm_b).round(4)}")
print(f"Decision Tree test accuracy: {d_arr.sum() / n_arr.sum():.4f}   95% CI {ci(dt_b).round(4)}")
lo, hi = ci(diff_b)
print(f"SVM minus Decision Tree:     {(s_arr.sum() - d_arr.sum()) / n_arr.sum():+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]")
if lo > 0:
    print("-> the interval is entirely above 0: SVM's advantage is unlikely to be chance.")
elif hi < 0:
    print("-> the interval is entirely below 0: the Decision Tree is genuinely better.")
else:
    print("-> the interval includes 0: the two models cannot be reliably told apart on this test set.")
RESULTS['svm_ci'], RESULTS['diff_ci'] = ci(svm_b), (lo, hi)

SVM test accuracy:           0.9343   95% CI [0.9261 0.942 ]
Decision Tree test accuracy: 0.9141   95% CI [0.906  0.9221]
SVM minus Decision Tree:     +0.0201   95% CI [+0.0139, +0.0258]
-> the interval is entirely above 0: SVM's advantage is unlikely to be chance.


## Step 11 -- Looking at what the model gets wrong

The confusion matrix shows _which_ classes get mixed up, not _why_. This step looks at a
misclassified signal and compares its feature values with what is typical for its true and
predicted classes -- a check that tells you whether errors make sense (two genuinely
similar artifact types) or something stranger.


In [18]:
wrong_idx = np.where(best_pred != y_test)[0]
print(f"{len(wrong_idx)} of {len(y_test)} test examples misclassified ({len(wrong_idx) / len(y_test):.1%})\n")

if len(wrong_idx) > 0:
    confusion_pairs = pd.Series([f"{y_test[i]} -> predicted {best_pred[i]}" for i in wrong_idx]).value_counts()
    print("Most common mistakes:")
    print(confusion_pairs.head(5))

    top_confusion = confusion_pairs.index[0]
    true_cls, pred_cls = top_confusion.split(" -> predicted ")
    example_idx = [i for i in wrong_idx if y_test[i] == true_cls and best_pred[i] == pred_cls][0]
    print(f"\nExample: a '{true_cls}' signal that got misclassified as '{pred_cls}'")
    comparison = pd.DataFrame({
        "This signal's value": X_test_feat[example_idx],
        f"Typical for {true_cls}": X_test_feat[y_test == true_cls].mean(axis=0),
        f"Typical for {pred_cls}": X_test_feat[y_test == pred_cls].mean(axis=0),
    }, index=FEATURE_NAMES)
    print(comparison.round(3))
    print(f"\nWhere 'This signal's value' sits closer to the {pred_cls} column than the {true_cls} column, "
          f"that explains the guess -- a borderline case rather than an arbitrary mistake.")
else:
    print("No misclassifications in the test set.")

1000 of 15210 test examples misclassified (6.6%)

Most common mistakes:
Baseline_Wander -> predicted Ambient_Light          181
Clean -> predicted Ambient_Light                    166
Powerline_Interference -> predicted Muscle_Noise    122
Muscle_Noise -> predicted Powerline_Interference    116
Ambient_Light -> predicted Baseline_Wander          105
Name: count, dtype: int64

Example: a 'Baseline_Wander' signal that got misclassified as 'Ambient_Light'
                    This signal's value  Typical for Baseline_Wander  Typical for Ambient_Light
Kurtosis                         -0.805                       -0.738                     -0.578
Skewness                         -0.038                       -0.007                     -0.006
AmplitudeEntropy                  4.585                        4.778                      4.610
ZeroCrossingRate                  0.056                        0.033                      0.038
AutocorrLag1                      0.964                        

## Step 12 -- Is this fast enough for a real device?

Accuracy alone does not say whether this could run on a wearable or phone in real time.
This measures how long it takes to classify and clean one signal.


In [19]:
n_timing_runs = 200
timing_idx = np.random.default_rng(0).choice(len(X_test_raw), size=min(n_timing_runs, len(X_test_raw)), replace=False)

feature_times, predict_times, clean_times = [], [], []
for i in timing_idx:
    sig = X_test_raw[i]
    t0 = time.perf_counter()
    feat = extract_sqi_features(sig)
    t1 = time.perf_counter()
    pred = best_model.predict(scaler.transform(np.array(feat).reshape(1, -1)))[0]
    t2 = time.perf_counter()
    _ = clean_ppg(sig, pred)
    t3 = time.perf_counter()
    feature_times.append((t1 - t0) * 1000); predict_times.append((t2 - t1) * 1000); clean_times.append((t3 - t2) * 1000)

total_ms = np.mean(feature_times) + np.mean(predict_times) + np.mean(clean_times)
signal_duration_ms = (SEG_LEN / FS) * 1000
print(f"Average time per signal (over {len(timing_idx)} runs):")
print(f"  Feature extraction: {np.mean(feature_times):.2f} ms")
print(f"  Classification:     {np.mean(predict_times):.2f} ms")
print(f"  Cleaning:           {np.mean(clean_times):.2f} ms")
print(f"  TOTAL:              {total_ms:.2f} ms")
print(f"\nEach signal covers {signal_duration_ms:.0f} ms of real time ({SEG_LEN / FS:.0f} seconds of recording).")
print(f"Processing takes {total_ms / signal_duration_ms:.2%} of the signal's own duration, which means "
      f"{'this keeps up comfortably with real-time data.' if total_ms < signal_duration_ms else 'this would fall behind real-time data.'}")
print("(Measured on this computer, not a wearable's processor -- a real device would likely be slower.)")
RESULTS['runtime_ms'] = total_ms

Average time per signal (over 200 runs):
  Feature extraction: 1.42 ms
  Classification:     1.34 ms
  Cleaning:           13.14 ms
  TOTAL:              15.90 ms

Each signal covers 16000 ms of real time (16 seconds of recording).
Processing takes 0.10% of the signal's own duration, which means this keeps up comfortably with real-time data.
(Measured on this computer, not a wearable's processor -- a real device would likely be slower.)


## Summary -- what this notebook shows (numbers are printed by the next cell)


In [20]:
print("=" * 78)
print("HEADLINE RESULTS FROM THIS RUN")
print("=" * 78)
print("\nData pool:", RESULTS['pool_by_source'])

r = RESULTS[{"SVM": 'svm', "DecisionTree": 'dt'}[RESULTS['best_name']]]
lo_c, hi_c = RESULTS['svm_ci']
print(f"\n1) Recognising synthetic artifact types on NEW recording groups ({RESULTS['best_name']}):")
print(f"   grouped cross-validation macro-F1 {r['cv']:.3f} | test accuracy {r['test_acc']:.3f} | "
      f"SVM 95% CI [{lo_c:.3f}, {hi_c:.3f}]")
print("   Per source:", {k: round(v, 3) for k, v in RESULTS['per_source_test_acc'].items()})

print(f"\n2) Overfitting check: train-minus-test accuracy = {RESULTS['overfit_gap']:+.3f} -> {RESULTS['overfit_verdict']}")
print(f"   learning-curve gap at full size = {RESULTS['learning_curve_gap']:+.3f}")

if 'loso' in RESULTS:
    print(f"\n3) Unseen data SOURCE (leave-one-source-out): mean accuracy {RESULTS['loso_mean_acc']:.3f}")
    print(RESULTS['loso'][['accuracy_on_unseen_source', 'clean_recall', 'normal_test_accuracy_same_source']].to_string())

print("\n4) Cleaning (primary comparison, same reference before/after):")
for cls, j in RESULTS['restoration'].items():
    vals = list(j.values())
    tag = f"{vals.count('better')} better, {vals.count('same')} unchanged, {vals.count('worse')} worse (of 3 metrics)"
    print(f"   {cls:24s} {tag}")

print("\n5) Real expert-labeled recordings (BUT PPG):")
A, B = RESULTS['external_A'], RESULTS['external_B']
if A is None:
    print("   not run (BUT PPG data or labels not available)")
else:
    print(f"   Part A (artifact-trained model): balanced accuracy {A['balanced_accuracy']:.1%} "
          f"[sensitivity {A['sensitivity']:.1%}, specificity {A['specificity']:.1%}] (50% = guessing)")
if B is not None:
    print(f"   Part B (7 features, trained on real labels): balanced accuracy {B['balanced_accuracy']:.1%}, AUC {B['auc']:.3f}")

print(f"\n6) Speed: {RESULTS['runtime_ms']:.1f} ms per 16-second signal on this computer.")

HEADLINE RESULTS FROM THIS RUN

Data pool: {'original_csv': 2576, 'but_ppg': 795, 'cuffless_bp': 600, 'ppg_dalia': 564}

1) Recognising synthetic artifact types on NEW recording groups (SVM):
   grouped cross-validation macro-F1 0.932 | test accuracy 0.934 | SVM 95% CI [0.926, 0.942]
   Per source: {'original_csv': 0.983, 'cuffless_bp': 0.868, 'ppg_dalia': 0.756, 'but_ppg': 0.974}

2) Overfitting check: train-minus-test accuracy = +0.024 -> moderate gap -> mild overfitting
   learning-curve gap at full size = +0.033

3) Unseen data SOURCE (leave-one-source-out): mean accuracy 0.412
                 accuracy_on_unseen_source  clean_recall  normal_test_accuracy_same_source
held_out_source                                                                           
original_csv                        0.5027        0.0000                            0.9830
cuffless_bp                         0.4014        0.0050                            0.8681
ppg_dalia                           0.3877     

### How to read these results honestly

- **In-distribution accuracy (item 1) is not real-world accuracy.** Test signals are made by
  the same noise generators as the training signals, so a high score shows the model learned
  those generators' signatures. Items 3 and 5 are the real tests of transfer; report them
  next to item 1, not instead of it.
- **Overfitting (item 2)** is judged from train-vs-test gap and the learning curve on
  held-out _recording groups_. A small gap says the model is not memorizing; it does not
  say the model transfers to real noise.
- **Cleaning (item 4)** is reported against the same reference before and after, so damage
  to the true heartbeat shape counts. A class that is not "better" on all 3 metrics is a
  real, reportable limitation, not something to hide. "Unchanged" for the Clean class is the
  desired outcome.

### Honest limitations

- **Real noise is not synthetic noise.** The classifier is trained only on artifacts we
  generated. Step 9 measures how far that carries to real recordings; Part B shows whether
  the features themselves are informative when real labels are available.
- **The original CSV has no volunteer information**, so each of its rows is treated as its
  own group. If several rows come from the same person, the test accuracy on that source
  can still be slightly optimistic.
- **"Clean" training data is assumed clean.** Signals from the public datasets are treated
  as artifact-free (after the PPG-DaLiA sitting-only filter, BUT's expert good-quality
  filter, and a basic sanity screen), but datasets without quality labels may still contain
  some real artifacts.
- **Few independent volunteers in the test set** (for example PPG-DaLiA has 15 subjects, of
  which about 3 are held out), so the confidence intervals are somewhat optimistic.
- **Cross-device generalization is only partly tested.** Leave-one-source-out (Step 5c) is a
  step towards it, but it assumes the new source's scale factor can be measured, and four
  public datasets are not every sensor type (finger clip, wrist, phone camera).
- **Runtime was measured on this computer, not on wearable hardware** (Step 12).
- **Cuff-Less BP is subsampled** (a few segments per record, capped overall) so its very long
  recordings do not dominate the pool.
